# ElasticSearch Test

In [36]:
from elasticsearch import Elasticsearch
import pandas as pd
import numpy as np
import datetime
import keras
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model

Create Elasticsearch queue-prediction index

In [7]:
# ignore 400 cause by IndexAlreadyExistsException when creating an index
es = Elasticsearch()
# es = Elasticsearch(
#       ['localhost'],
#       http_auth=(username, 'password'),
#       verify_certs=False,
#       scheme="https",
#       port=443,
# )
es.indices.create(index='queues-prediction', ignore=400) #can be ignored

{'error': {'root_cause': [{'type': 'resource_already_exists_exception',
    'reason': 'index [queues-prediction/D3O1OrPrRqGFs1v3O06EAw] already exists',
    'index_uuid': 'D3O1OrPrRqGFs1v3O06EAw',
    'index': 'queues-prediction'}],
  'type': 'resource_already_exists_exception',
  'reason': 'index [queues-prediction/D3O1OrPrRqGFs1v3O06EAw] already exists',
  'index_uuid': 'D3O1OrPrRqGFs1v3O06EAw',
  'index': 'queues-prediction'},
 'status': 400}

### Load from Queue Index

Match products queue

    "gte" : "now-1d/d", (yesterday)
    "lt" :  "now/d"     (today)

In [37]:
res = es.search(index="queues", body={"query" : {
                                        "bool" : { 
                                          "must" : [
                                            {"match": {
                                                    "name" : "products"}},
                                            {"range": {
                                                    "timestamp":{
                                                        "gte": "2019-12-29",
                                                        "lte": "2019-12-31",
                                                        "format": "yyyy-MM-dd"
                                                  }
                                                }
                                              }
                                          ]
                                        }
                                      }
                                    }, size=1000) #define size

Get _source Data

In [38]:
d = [elem['_source'] for elem in res['hits']['hits']]

In [39]:
for elem in d:
    del elem['items']
    del elem['querytime']

Build Dataframe

In [40]:
df = pd.DataFrame(d)

In [41]:
df.index = df["timestamp"]

In [42]:
df.index = pd.to_datetime(df.index, format='%Y-%m-%dT%H:%M:%S.%f%z').sort_values()

In [43]:
df.drop(columns=['timestamp', 'name', 'tier'], inplace=True)

Resample Data to 1min Interval 

In [44]:
df = df.resample('15T').mean().ffill()

In [45]:
df = df.fillna(0)

In [46]:
df.head()

,size
timestamp,
2019-12-29 21:15:00+00:00,32671.875
2019-12-29 21:30:00+00:00,32671.875
2019-12-29 21:45:00+00:00,32671.875
2019-12-29 22:00:00+00:00,32671.875
2019-12-29 22:15:00+00:00,32671.875


## ML Model

In [47]:
from tensorflow.keras.models import load_model

In [48]:
model = load_model('model_lstm_15min_woi.h5')

In [49]:
train = df.values

In [50]:
scaler = MinMaxScaler()
scaler.fit(train)
train = scaler.transform(train)

In [51]:
n_input = 96
n_features = 1

In [52]:
pred_list = []

batch = train[-n_input:].reshape((1, n_input, n_features)) #first batch is the last day in

# for 96 steps, always predict 1 step ahead and add that to the pred_list and update the batch with it
for i in range(n_input):   
    pred_list.append(model.predict(batch)[0]) 
    batch = np.append(batch[:,1:,:],[[pred_list[i]]],axis=1)

In [53]:
pred = np.array(scaler.inverse_transform(pred_list)).reshape(-1)

In [54]:
pred = [int(x) for x in pred]

In [55]:
time_stamps = pd.date_range(start=df.index[-1]+datetime.timedelta(minutes=15), periods=96, freq='15min',tz='utc')

In [56]:
d = {'timestamp': time_stamps, 'size': pred}
pred_df = pd.DataFrame(data=d)

In [57]:
pred_df

,timestamp,size
0,2019-12-31 04:15:00+00:00,28882
1,2019-12-31 04:30:00+00:00,28883
2,2019-12-31 04:45:00+00:00,28876
3,2019-12-31 05:00:00+00:00,28866
4,2019-12-31 05:15:00+00:00,28854
...,...,...
91,2020-01-01 03:00:00+00:00,27677
92,2020-01-01 03:15:00+00:00,27666
93,2020-01-01 03:30:00+00:00,27655
94,2020-01-01 03:45:00+00:00,27645


In [58]:
count = 0
for index, row in pred_df.iterrows():
    doc_data = {
        'timestamp': row['timestamp'],
        'tier' : 'pic',
        'name' : 'products',
    #     'querytime' : 0,
        'size' : row['size'],
    #     'items' : " ".join(items)
    }
    count += 1
    es.index('queues-prediction', body=doc_data)
    if count % 5 == 0:
        print(str(count) + " Elemente hochgeladen")

5 Elemente hochgeladen
10 Elemente hochgeladen
15 Elemente hochgeladen
20 Elemente hochgeladen
25 Elemente hochgeladen
30 Elemente hochgeladen
35 Elemente hochgeladen
40 Elemente hochgeladen
45 Elemente hochgeladen
50 Elemente hochgeladen
55 Elemente hochgeladen
60 Elemente hochgeladen
65 Elemente hochgeladen
70 Elemente hochgeladen
75 Elemente hochgeladen
80 Elemente hochgeladen
85 Elemente hochgeladen
90 Elemente hochgeladen
95 Elemente hochgeladen
